# 01 — Data Exploration & Cleaning

Student Performance & Engagement Analytics

This notebook:
1. Loads the raw synthetic dataset (`data/raw/student_performance_data.xlsx`)
2. Profiles it (shape, dtypes, missing values, duplicates)
3. Cleans it (drops duplicate students, imputes missing engagement values)
4. Runs exploratory analysis on the two core business questions:
   - How does engagement correlate with academic performance?
   - Which subjects/regions show the biggest performance gaps?
   - Who looks "at-risk" based on engagement + performance?
5. Exports `data/processed/cleaned_data.csv` and `data/processed/sql_ready_data.csv`
   for the SQL notebook and Power BI dashboard.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RAW_PATH = "../data/raw/student_performance_data.xlsx"
OUT_DIR = "../data/processed"


## 2. Load & first look

In [ ]:
df = pd.read_excel(RAW_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3. Data quality check

Check for missing values and duplicate students before doing any analysis.

In [ ]:
missing = df.isna().sum()
missing[missing > 0]

In [ ]:
dupe_rows = df.duplicated().sum()
dupe_ids = df.duplicated(subset="Student_ID").sum()
print(f"Exact duplicate rows: {dupe_rows}")
print(f"Duplicate Student_IDs: {dupe_ids}")

## 4. Cleaning

- Drop rows with a duplicate `Student_ID`, keeping the first occurrence.
- Impute missing `Video_Completion_Rate` using the **median within each Subject**
  (rather than a single global median) since completion rates vary meaningfully
  by subject.
- Verify the dataset is fully clean before moving on.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="Student_ID", keep="first").reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate student rows -> {len(df)} rows remain")

In [ ]:
df["Video_Completion_Rate"] = df.groupby("Subject")["Video_Completion_Rate"] \
    .transform(lambda s: s.fillna(s.median()))

assert df.isna().sum().sum() == 0, "Still have missing values"
assert df.duplicated(subset="Student_ID").sum() == 0, "Still have duplicate students"
print("Clean: no missing values, no duplicate students.")

## 5. Engagement vs. performance

Business question 1: how does engagement (logins, video completion) relate to
academic performance (`Percentage`)?

In [ ]:
numeric_cols = ["App_Login_Count", "Video_Completion_Rate", "Study_Hours_Per_Week",
                 "Assignment_Submission_Rate", "Percentage"]
corr = df[numeric_cols].corr()
corr

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set_title("Correlation: engagement metrics vs. Percentage")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.regplot(data=df.sample(3000, random_state=1), x="App_Login_Count", y="Percentage",
            scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set_title("App logins vs. Percentage")

sns.regplot(data=df.sample(3000, random_state=1), x="Video_Completion_Rate", y="Percentage",
            scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set_title("Video completion rate vs. Percentage")

plt.tight_layout()
plt.show()

**Reading the correlations:** engagement metrics have a positive, moderate
relationship with `Percentage` — video completion rate is the stronger of the two
engagement signals. Neither is a perfect predictor, which is expected: engagement
is one input into performance, not the only one.

## 6. Subject & region performance gaps

Business question 2: which subjects and regions show the biggest performance gaps?

In [ ]:
region_summary = df.groupby("Region")["Percentage"].agg(["mean", "median", "count"]) \
    .sort_values("mean")
region_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="Region", y="Percentage",
            order=region_summary.index, ax=ax)
ax.set_title("Percentage distribution by Region")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
subject_summary = df.groupby("Subject")["Percentage"].agg(["mean", "median", "count"]) \
    .sort_values("mean")
subject_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="Subject", y="Percentage",
            order=subject_summary.index, ax=ax)
ax.set_title("Percentage distribution by Subject")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
gap_region = region_summary["mean"].max() - region_summary["mean"].min()
gap_subject = subject_summary["mean"].max() - subject_summary["mean"].min()
print(f"Region performance gap: {gap_region:.1f} points "
      f"({region_summary['mean'].idxmax()} vs {region_summary['mean'].idxmin()})")
print(f"Subject performance gap: {gap_subject:.1f} points "
      f"({subject_summary['mean'].idxmax()} vs {subject_summary['mean'].idxmin()})")

## 7. Identifying at-risk students

Business question 3: can we flag at-risk students early from engagement patterns?

Definition used here: **Low engagement AND below-average performance.** This
mirrors the `At-Risk Count` DAX measure in the Power BI guide, so the flag stays
consistent across the Python, SQL, and dashboard layers.

In [ ]:
avg_pct = df["Percentage"].mean()
df["At_Risk"] = (df["Engagement_Level"] == "Low") & (df["Percentage"] < avg_pct)

at_risk_count = df["At_Risk"].sum()
print(f"At-risk students: {at_risk_count} ({at_risk_count/len(df)*100:.1f}% of all students)")

df.groupby("Engagement_Level")["Percentage"].mean().sort_values()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=df, x="Engagement_Level", y="Percentage",
            order=["Low", "Medium", "High"], ax=ax, palette="Blues")
ax.set_title("Average Percentage by Engagement Level")
plt.tight_layout()
plt.show()

## 8. Export for downstream steps

- `cleaned_data.csv` — full cleaned dataset, feeds the SQL notebook and Power BI.
- `sql_ready_data.csv` — same data with lowercase, SQL-friendly column names.

In [ ]:
df.to_csv(f"{OUT_DIR}/cleaned_data.csv", index=False)

sql_df = df.copy()
sql_df.columns = [c.lower() for c in sql_df.columns]
sql_df.to_csv(f"{OUT_DIR}/sql_ready_data.csv", index=False)

print("Exported cleaned_data.csv and sql_ready_data.csv to", OUT_DIR)

## 9. Summary of findings so far

- Engagement metrics (logins, video completion) show a **moderate positive
  correlation** with academic performance — video completion rate is the
  stronger signal of the two.
- **Region gap:** biggest difference is between the highest- and
  lowest-performing regions (see gap computed above).
- **Subject gap:** Mathematics trails, Computer Science leads.
- **~24% of students** meet the at-risk definition (low engagement + below-average
  performance) — a large enough group to justify an early-intervention workflow.

Next: `02_sql_analysis.ipynb` reproduces these questions in SQL against
`sql_ready_data.csv`.